# SCaDa Kestirimci Bakim - EDA (Kesifsel Veri Analizi)
## Kisi 1: AI & Veri Bilimi

Bu notebook, NASA C-MAPSS FD001 verisi uzerinde kesifsel veri analizi yapar.
- Sensör trendlerini inceler
- Bozulma desenlerini tespit eder
- Kayan ortalama ozellikleri olusturur
- Temizlenmis veri setini hazirlar

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Grafik ayarlar
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {plt.matplotlib.__version__}")

## 1. Veri Yukleme

In [ ]:
# Dinamik sutun isimleri olustur (her C-MAPSS versiyonunda calisir)
first_line = open('data/FD001/train.txt').readline().split()
num_cols = len(first_line)  # FD001 icin 26
col_names = ['engine_id', 'time_in_cycles',
             'op_setting_1', 'op_setting_2', 'op_setting_3']
for i in range(1, num_cols - 5 + 1):  # 26 - 5 = 21 sensör
    col_names.append(f'sensor_{i}')

print(f"Toplam sutun: {len(col_names)}")
print(f"Sensörler: {col_names[5:]}")

# Egitim verisini yukle
train = pd.read_csv('data/FD001/train.txt', sep='\s+', header=None)
train.columns = col_names

# RUL verisini yukle ve birlestir
rul = pd.read_csv('data/FD001/RUL.txt', sep='\s+', header=None)
rul.columns = ['engine_id', 'RUL']
train = train.merge(rul, on='engine_id')

print(f"\nVeri seti sekli: {train.shape}")
print(f"\nIlk 5 satir:")
train.head()

## 2. Temel Istatistikler

In [ ]:
# Genel istatistikler
print("=== Genel Istatistikler ===")
print(train.describe())

# Eksik deger kontrolu
print(f"\n=== Eksik Degerler ===")
print(train.isnull().sum().sum(), "toplam eksik deger")

# Motor istatistikleri
num_engines = train['engine_id'].nunique()
print(f"\n=== Motor Bilgileri ===")
print(f"Toplam motor sayisi: {num_engines}")
print(f"Dongu sayisi: {len(train)}")
print(f"Ortalama dongu/motor: {len(train)/num_engines:.0f}")

# RUL istatistikleri
print(f"\n=== RUL Istatistikleri ===")
print(train['RUL'].describe())

## 3. RUL Dagilimi

In [ ]:
plt.figure(figsize=(14, 6))
plt.hist(train['RUL'], bins=100, edgecolor='black', color='steelblue')
plt.title('RUL Dagilimi (Tum Motorlar)', fontsize=14)
plt.xlabel('Kalan有用 Omur (dongu)', fontsize=12)
plt.ylabel('Motor Sayisi', fontsize=12)
plt.axvline(train['RUL'].median(), color='red', linestyle='--', label=f"Medyan: {train['RUL'].median():.0f}")
plt.legend()
plt.tight_layout()
plt.savefig('rul_distribution.png', dpi=150)
plt.show()
print("Grafik kaydedildi: rul_distribution.png")

## 4. Sensör Trendlerini Görsellestirme

In [ ]:
# Bir motor secip tum sensörleri dongu boyunca ciz
engine_id = train['engine_id'].iloc[0]
engine_data = train[train['engine_id'] == engine_id].sort_values('time_in_cycles')

print(f"Motor #{engine_id} - Toplam {len(engine_data)} dongu")
print(f"Son RUL degeri: {engine_data['RUL'].iloc[-1]}")

# Sensör sutunlarini tespit et
sensor_cols = [col for col in train.columns if col.startswith('sensor_')]
print(f"Tespit edilen sensör sayisi: {len(sensor_cols)}")

In [ ]:
# Tum sensörleri subplot olarak ciz
num_sensors = len(sensor_cols)
n_rows = (num_sensors + 2) // 3  # Her satirda max 3 sensör

plt.figure(figsize=(16, 4 * n_rows))
for idx, sensor in enumerate(sensor_cols, 1):
    plt.subplot(n_rows, 3, idx)
    plt.plot(engine_data['time_in_cycles'], engine_data[sensor], linewidth=0.8, color='steelblue')
    plt.title(f'{sensor}', fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.xlabel('Dongu')
plt.suptitle(f'Motor #{engine_id} - Sensör Trendleri (Tum Hayat Boyu)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('sensor_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafik kaydedildi: sensor_trends.png")

## 5. Bozulma Desenlerini Tespit Etme

In [ ]:
# Ilk 50 dongu vs son 50 dongu karsilastirmasi
early = engine_data.head(50).mean()
late = engine_data.tail(50).mean()
diff = late - early

plt.figure(figsize=(14, 6))
num_sensors = len(sensor_cols)
colors = ['red' if v < 0 else 'green' for v in diff.values]
plt.bar(range(num_sensors), diff.values, edgecolor='black', color=colors)
plt.xticks(range(num_sensors), [f'S{i+1}' for i in range(num_sensors)], rotation=45)
plt.title('Sensör Bozulmasi: Baslangic vs Son (Ortalama Degisim)', fontsize=13)
plt.xlabel('Sensör Index', fontsize=12)
plt.ylabel('Ortalama Degisim (Son - Baslangic)', fontsize=12)
plt.axhline(y=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('sensor_degradation.png', dpi=150)
plt.show()
print("Grafik kaydedildi: sensor_degradation.png")

In [ ]:
# En cok degisen sensörleri listeleyin
diff_sorted = diff.sort_values(ascending=False)
print("En cok degisen 10 sensör (artan):")
print(diff_sorted.head(10).to_string())
print("\nEn cok degisen 10 sensör (azalan):")
print(diff_sorted.tail(10).to_string())

## 6. Birkaç Motoru Karilastiralim

In [ ]:
# Farkli motorlari karsilastir - saglikli vs hizli bozulan
engine_ids = sorted(train['engine_id'].unique())
short_life_id = train.groupby('engine_id')['RUL'].last().idxmin()
long_life_id = train.groupby('engine_id')['RUL'].last().idxmax()

print(f"En kisa omur: Motor #{short_life_id} (son RUL: {train[train['engine_id']==short_life_id].sort_values('time_in_cycles').tail(1)['RUL'].values[0]})")
print(f"En uzun omur: Motor #{long_life_id} (son RUL: {train[train['engine_id']==long_life_id].sort_values('time_in_cycles').tail(1)['RUL'].values[0]})")

short_data = train[train['engine_id'] == short_life_id].sort_values('time_in_cycles')
long_data = train[train['engine_id'] == long_life_id].sort_values('time_in_cycles')

# RUL karsilastirmasi
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(short_data['time_in_cycles'], short_data['RUL'], color='red', linewidth=1.5)
plt.title(f'Motor #{short_life_id} - Kisa Omur')
plt.xlabel('Dongu')
plt.ylabel('RUL')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(long_data['time_in_cycles'], long_data['RUL'], color='green', linewidth=1.5)
plt.title(f'Motor #{long_life_id} - Uzun Omur')
plt.xlabel('Dongu')
plt.ylabel('RUL')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rul_comparison.png', dpi=150)
plt.show()
print("Grafik kaydedildi: rul_comparison.png")

## 7. Ozellik Muhendisligi - Kayan Ortalamalar

In [ ]:
def add_rolling_features(df, window=50):
    """Her sensör icin kayan ortalama ve standart sapma ekle."""
    df = df.copy()
    sensor_cols = [col for col in df.columns if col.startswith('sensor_')]
    
    for engine_id, group in df.groupby('engine_id'):
        for sensor in sensor_cols:
            rolling_mean = group[sensor].rolling(window=window, min_periods=1).mean()
            rolling_std = group[sensor].rolling(window=window, min_periods=1).std()
            df.loc[group.index, f'{sensor}_rmean'] = rolling_mean
            df.loc[group.index, f'{sensor}_rstd'] = rolling_std
    return df

train_eng = add_rolling_features(train, window=50)
print(f"Ozellik sayisi: {len(train_eng.columns)} (baslangicta {len(train.columns)})")
print(f"Yeni sutunlar: {[c for c in train_eng.columns if c not in train.columns][:10]}...")

In [ ]:
def add_difference_features(df):
    """Her sensör icin bir onceki dongudeki degerden farki hesapla."""
    df = df.copy()
    sensor_cols = [col for col in df.columns if col.startswith('sensor_')]
    
    for engine_id, group in df.groupby('engine_id'):
        for sensor in sensor_cols:
            df.loc[group.index, f'{sensor}_diff'] = group[sensor].diff().fillna(0)
    return df

train_eng = add_difference_features(train_eng)
print(f"Toplam ozellik sayisi: {len(train_eng.columns)}")

## 8. Farkli Pencere Boyutlari Denemesi

In [ ]:
# 3 farkli pencere boyutunda kayan ortalama olustur
windows = [10, 30, 100]

for w in windows:
    temp_df = add_rolling_features(train.copy(), window=w)
    print(f"Window={w}: {len(temp_df.columns)} ozellik")
    
    # Bir sensör icin ornek cikar
    sample = temp_df[temp_df['engine_id'] == 1].head(60)
    if w == 30:
        print(f"\nOrnek (Motor #1, sensor_1, window={w}):")
        print(sample[['time_in_cycles', 'sensor_1', 'sensor_1_rmean', 'sensor_1_rstd']].to_string(index=False))

print("\nNot: Window=30 baslangic icin iyi bir denge sunar.")
print("  Window=10: Daha hassas, daha gürültülü")
print("  Window=100: Daha düz, gec tepki")

## 9. Correlation Matrix - Sensör Iliskileri

In [ ]:
# RUL ile en yüksek korelasyona sahip sensörleri bul
sensor_cols = [col for col in train_eng.columns if col.startswith('sensor_')]

correlations = []
for sensor in sensor_cols:
    corr = train_eng[sensor].corr(train_eng['RUL'])
    correlations.append((sensor, corr))

correlations.sort(key=lambda x: abs(x[1]), reverse=True)

print("RUL ile en yüksek korelasyona sahip sensörler:")
for name, corr in correlations[:10]:
    print(f"  {name}: {corr:.4f}")

# Korelasyon grafigi (ilk 10 sensör)
top_sensors = [c[0] for c in correlations[:10]]
corr_matrix = train_eng[['RUL'] + top_sensors].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('RUL ile En Iliskili 10 Sensör - Korelasyon Matrisi', fontsize=13)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150)
plt.show()
print("Grafik kaydedildi: correlation_matrix.png")

## 10. Temizlenmis Veriyi Kaydet

In [ ]:
# Final feature set
feature_cols = ['time_in_cycles', 'op_setting_1', 'op_setting_2', 'op_setting_3'] + \
               sensor_cols + \
               [c for c in train_eng.columns if 'rmean' in c or 'rstd' in c] + \
               [c for c in train_eng.columns if '_diff' in c]

print(f"Toplam feature: {len(feature_cols)}")
print(f"Feature listesi:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:3d}. {col}")

# Veriyi kaydet (sonraki hafta model egitimi icin)
train_eng.to_csv('data/train_engineered.csv', index=False)
print(f"\nÖzellikler eklendi veri kaydedildi: data/train_engineered.csv")
print(f"Sekil: {train_eng.shape}")
print(f"Features: {len(feature_cols)}")